In [1]:
!nvidia-smi

Mon Jul  6 17:54:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers
!pip install -q datasets
!pip install -q peft
!pip install -q trl
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.2 MB/s eta 0:00:00


In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "Anupam007/indian-recipe-dataset",
    split="train"
)

Cleaned_Indian_Food_Dataset.csv:   0%|          | 0.00/11.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5938 [00:00<?, ? examples/s]

In [6]:
print(dataset)

Dataset({
    features: ['TranslatedRecipeName', 'TranslatedIngredients', 'TotalTimeInMins', 'Cuisine', 'TranslatedInstructions', 'URL', 'Cleaned-Ingredients', 'image-url', 'Ingredient-count'],
    num_rows: 5938
})


In [7]:
print(dataset.features)

{'TranslatedRecipeName': Value('string'), 'TranslatedIngredients': Value('string'), 'TotalTimeInMins': Value('int64'), 'Cuisine': Value('string'), 'TranslatedInstructions': Value('string'), 'URL': Value('string'), 'Cleaned-Ingredients': Value('string'), 'image-url': Value('string'), 'Ingredient-count': Value('int64')}


In [8]:
dataset[0]

{'TranslatedRecipeName': 'Masala Karela Recipe',
 'TranslatedIngredients': '1 tablespoon Red Chilli powder,3 tablespoon Gram flour (besan),2 teaspoons Cumin seeds (Jeera),1 tablespoon Coriander Powder (Dhania),2 teaspoons Turmeric powder (Haldi),Salt - to taste,1 tablespoon Amchur (Dry Mango Powder),6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Sunflower Oil - as required,1 Onion - thinly sliced',
 'TotalTimeInMins': 45,
 'Cuisine': 'Indian',
 'TranslatedInstructions': 'To begin making the Masala Karela Recipe,de-seed the karela and slice.\nDo not remove the skin as the skin has all the nutrients.\nAdd the karela to the pressure cooker with 3 tablespoon of water, salt and turmeric powder and pressure cook for three whistles.\nRelease the pressure immediately and open the lids.\nKeep aside.Heat oil in a heavy bottomed pan or a kadhai.\nAdd cumin seeds and let it sizzle.Once the cumin seeds have sizzled, add onions and saute them till it turns golden brown in color.Add the karela, red chi

teaching the llm to how to take data as it cant take row and columns (user and assistent)

In [10]:
recipe = dataset[0]

for key, value in recipe.items():
    print("=" * 50)
    print(key)#key value pairs like dictionary in python
    print(value)

TranslatedRecipeName
Masala Karela Recipe
TranslatedIngredients
1 tablespoon Red Chilli powder,3 tablespoon Gram flour (besan),2 teaspoons Cumin seeds (Jeera),1 tablespoon Coriander Powder (Dhania),2 teaspoons Turmeric powder (Haldi),Salt - to taste,1 tablespoon Amchur (Dry Mango Powder),6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Sunflower Oil - as required,1 Onion - thinly sliced
TotalTimeInMins
45
Cuisine
Indian
TranslatedInstructions
To begin making the Masala Karela Recipe,de-seed the karela and slice.
Do not remove the skin as the skin has all the nutrients.
Add the karela to the pressure cooker with 3 tablespoon of water, salt and turmeric powder and pressure cook for three whistles.
Release the pressure immediately and open the lids.
Keep aside.Heat oil in a heavy bottomed pan or a kadhai.
Add cumin seeds and let it sizzle.Once the cumin seeds have sizzled, add onions and saute them till it turns golden brown in color.Add the karela, red chilli powder, amchur powder, coriander

take the required ingredients

In [11]:
def create_prompt(example):

    prompt = f"""
User:
How do I make {example['TranslatedRecipeName']}?

Assistant:
Ingredients:
{example['TranslatedIngredients']}

Instructions:
{example['TranslatedInstructions']}
"""

    return {"text": prompt}

If the dataset has

6000 recipes

this function will be called

6000 times.

so we are applying to entire dataset.

In [12]:
dataset = dataset.map(create_prompt)

Map:   0%|          | 0/5938 [00:00<?, ? examples/s]

In [15]:
print(dataset[15]["text"])


User:
How do I make South Indian Onion Chutney Recipe - South Indian Onion Chutney (Recipe)?

Assistant:
Ingredients:
 1 teaspoon cumin seeds, 2 teaspoons oil, 2 tablespoons black urad dal (split), salt - 1 teaspoon, 3 dry red chillies, 1/2 teaspoon oil, 1 tablespoon tamarind paste, as per taste Rye, 1 sprig curry leaves, 1/2 teaspoon jaggery,2 onions

Instructions:
To make South Indian Onion Chutney, first of all chop the onion and keep it aside.
Now heat 1 teaspoon of oil in the pan.
Add cumin seeds, dry red chillies and let it cook for 10 seconds.
Now add urad dal in it and let it cook till it becomes golden.
Turn off the gas and drain in a bowl.
Add another spoon of oil to the same pan.
Add onions and let it cook for 4 to 5 minutes.
Turn off the gas and let it cool down.
Now put urad dal, cumin and dry red chillies in the mixer grinder and grind them.
Add cooked onions, tamarind and jaggery.
Add some water and grind it.
Now for the tempering, heat the oil in a small pan.
Add musta